# Notebook 08 — The Emotional Map (Lesion Study)

Systematic compression of each part of BERT to reveal where each emotion lives
inside the transformer. Inspired by neuroscience lesion studies: damage one
piece, observe what is lost.

**Experiments:**
- Per-layer lesion (12 experiments): compress all components of one layer to rank 128
- Per-component lesion (6 experiments): compress one component type across all 12 layers

**Visualizations:**
1. Emotional Map — dual heatmap (component x emotion + layer x emotion)
2. Emotional Genealogy — hierarchical clustering of emotions by sensitivity profile
3. Emotional Survival — extinction order under increasing compression
4. Emotional Brain Scan — per-emotion 2D maps of layer x component importance

**Structure:** Setup → Part A (lesion experiments) → Part B (sensitivity matrices) →
Part C (visualizations) → Part D (export).

## Setup

In [ ]:
import sys, os, warnings, time
warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/transformer-structural-compression-v2'
    IN_COLAB = True
except ImportError:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    IN_COLAB = False

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from scipy.cluster.hierarchy import linkage, dendrogram, leaves_list, fcluster, set_link_color_palette
from scipy.spatial.distance import pdist
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

from src.data import load_goemotions
from src.utils import compute_metrics
from src.compression import (
    apply_svd_compression,
    get_target_layer_names,
    filter_layer_names,
)

EXCLUDED_EMOTIONS = ['neutral', 'grief', 'nervousness', 'pride', 'relief']
LESION_RANK = 128

device = 'cuda' if torch.cuda.is_available() else 'cpu'

results_dir = os.path.join(PROJECT_ROOT, 'results')
out_dir = os.path.join(results_dir, 'notebook8')
os.makedirs(out_dir, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Device: {device}')
print(f'Output directory: {out_dir}')

In [ ]:
# Dark theme + neon palette (consistent with notebooks 05-07)
DARK_BG        = '#0d1117'
DARK_GRID      = '#21262d'
TEXT_PRIMARY   = '#e6edf3'
TEXT_SECONDARY = '#8b949e'

COMP_COLORS = ['#ff006e', '#ff9500', '#ffd60a', '#00ff87', '#00d4ff', '#7b2ff7']

NEON_CMAP = LinearSegmentedColormap.from_list(
    'neon', ['#00ff87', '#00d4ff', '#7b2ff7', '#ff006e'], N=256)

RETENTION_CMAP = LinearSegmentedColormap.from_list(
    'retention', ['#ff006e', '#ff4d6d', '#ff9500', '#ffd60a', '#80ffb4', '#00ff87'], N=256)

plt.rcParams.update({
    'figure.facecolor':  DARK_BG,
    'axes.facecolor':    DARK_BG,
    'text.color':        TEXT_PRIMARY,
    'axes.labelcolor':   TEXT_PRIMARY,
    'xtick.color':       TEXT_SECONDARY,
    'ytick.color':       TEXT_SECONDARY,
    'axes.edgecolor':    DARK_GRID,
    'grid.color':        DARK_GRID,
    'figure.dpi':        150,
    'savefig.dpi':       300,
    'savefig.facecolor': DARK_BG,
    'savefig.bbox':      'tight',
    'font.size':         12,
})
print('Dark theme configured')

In [ ]:
# Load fine-tuned model + dataset
model_path = os.path.join(PROJECT_ROOT, 'results', 'bert-goemotions-23emo-final')
baseline_model = AutoModelForSequenceClassification.from_pretrained(
    model_path, problem_type='multi_label_classification',
)
baseline_model.eval()
baseline_model.to(device)
num_labels = baseline_model.config.num_labels
print(f'Model loaded from {model_path}')
print(f'num_labels: {num_labels}')

_, _, test_ds, emotion_names, data_collator = load_goemotions(exclude_emotions=EXCLUDED_EMOTIONS)
print(f'Test set: {len(test_ds)} examples')

eval_args = TrainingArguments(
    output_dir=os.path.join(results_dir, 'tmp_eval'),
    per_device_eval_batch_size=64,
    fp16=(device == 'cuda'),
    report_to='none',
)

In [ ]:
# Helper for lesion experiments
all_target_names = get_target_layer_names(baseline_model)


def run_lesion_experiment(component=None, layers=None, rank=LESION_RANK):
    """Compress specific component/layers and return (per-emotion F1 array, F1 macro)."""
    target = filter_layer_names(all_target_names, component=component, layers=layers)
    # Workaround: 'ffn_output' pattern also matches attention.output.dense
    if component == 'ffn_output':
        target = [n for n in target if 'attention' not in n]
    model = apply_svd_compression(baseline_model, rank=rank, layer_names=target)
    model.to(device)
    trainer = Trainer(
        model=model, args=eval_args,
        compute_metrics=compute_metrics, data_collator=data_collator,
    )
    metrics = trainer.evaluate(test_ds)
    per_emotion_f1 = np.array([metrics[f'eval_f1_label_{i}'] for i in range(num_labels)])
    f1_macro = metrics['eval_f1_macro']
    del model, trainer
    if device == 'cuda':
        torch.cuda.empty_cache()
    return per_emotion_f1, f1_macro

## Part A — Lesion Experiments

Run the baseline, then two lesion batteries:
- 12 per-layer experiments
- 6 per-component experiments

In [ ]:
# Baseline evaluation
print('Evaluating baseline...')
baseline_trainer = Trainer(
    model=baseline_model, args=eval_args,
    compute_metrics=compute_metrics, data_collator=data_collator,
)
baseline_metrics = baseline_trainer.evaluate(test_ds)
baseline_per_emotion = np.array(
    [baseline_metrics[f'eval_f1_label_{i}'] for i in range(num_labels)]
)
baseline_f1_macro = baseline_metrics['eval_f1_macro']
del baseline_trainer

print(f'Baseline F1 macro: {baseline_f1_macro:.4f}')
print(f'Per-emotion range: [{baseline_per_emotion.min():.4f}, {baseline_per_emotion.max():.4f}]')

In [ ]:
# Per-layer lesion: compress ALL components of each layer to rank 128
layer_f1 = np.zeros((12, num_labels))
layer_f1_macro = np.zeros(12)

print(f'Per-layer lesion study (rank={LESION_RANK})')
print('=' * 55)
t_start = time.time()

for layer_idx in range(12):
    t0 = time.time()
    f1_arr, f1_m = run_lesion_experiment(layers=[layer_idx], rank=LESION_RANK)
    layer_f1[layer_idx] = f1_arr
    layer_f1_macro[layer_idx] = f1_m
    elapsed = time.time() - t0
    print(f'  Layer {layer_idx:2d}: F1 macro = {f1_m:.4f}  ({elapsed:.1f}s)')

print(f'\nCompleted 12 layer lesions in {time.time() - t_start:.0f}s')
print(f'Most sensitive layer: {np.argmin(layer_f1_macro)} '
      f'(F1 = {layer_f1_macro.min():.4f})')
print(f'Least sensitive layer: {np.argmax(layer_f1_macro)} '
      f'(F1 = {layer_f1_macro.max():.4f})')

In [ ]:
# Per-component lesion: compress each component type across ALL layers to rank 128
COMPONENT_KEYS   = ['query', 'key', 'value', 'attention_output', 'intermediate', 'ffn_output']
COMPONENT_LABELS = ['Query', 'Key', 'Value', 'Attn Out', 'FFN Inter', 'FFN Out']

component_f1 = np.zeros((6, num_labels))
component_f1_macro = np.zeros(6)

print(f'Per-component lesion study (rank={LESION_RANK})')
print('=' * 55)
t_start = time.time()

for i, comp in enumerate(COMPONENT_KEYS):
    t0 = time.time()
    f1_arr, f1_m = run_lesion_experiment(component=comp, rank=LESION_RANK)
    component_f1[i] = f1_arr
    component_f1_macro[i] = f1_m
    elapsed = time.time() - t0
    print(f'  {COMPONENT_LABELS[i]:12s}: F1 macro = {f1_m:.4f}  ({elapsed:.1f}s)')

print(f'\nCompleted 6 component lesions in {time.time() - t_start:.0f}s')
print(f'Most damaging component: {COMPONENT_LABELS[np.argmin(component_f1_macro)]} '
      f'(F1 = {component_f1_macro.min():.4f})')
print(f'Least damaging component: {COMPONENT_LABELS[np.argmax(component_f1_macro)]} '
      f'(F1 = {component_f1_macro.max():.4f})')

## Part B — Sensitivity Matrices

Compute F1 retention (compressed / baseline) per layer and per component,
filter emotions with non-trivial baseline, and order them by total vulnerability.

In [ ]:
# Retention = compressed / baseline (1.0 = no damage, 0.0 = total loss)
safe_baseline = np.where(baseline_per_emotion > 0, baseline_per_emotion, 1.0)

layer_retention     = layer_f1 / safe_baseline[np.newaxis, :]       # (12, num_labels)
component_retention = component_f1 / safe_baseline[np.newaxis, :]   # (6, num_labels)

# Clip to [0, 1] — retention above 1.0 can happen due to evaluation noise
layer_retention     = np.clip(layer_retention, 0.0, 1.0)
component_retention = np.clip(component_retention, 0.0, 1.0)

# Filter out emotions with baseline F1 ~ 0
valid_mask    = baseline_per_emotion > 0.01
valid_indices = np.where(valid_mask)[0]
valid_emotions = [emotion_names[i] for i in valid_indices]
n_valid = len(valid_emotions)

layer_ret_valid     = layer_retention[:, valid_indices]       # (12, n_valid)
component_ret_valid = component_retention[:, valid_indices]   # (6, n_valid)

# Sort emotions by total vulnerability (mean retention across all 18 lesions)
all_retention = np.concatenate([component_ret_valid, layer_ret_valid], axis=0)  # (18, n_valid)
mean_retention = all_retention.mean(axis=0)

sort_idx = np.argsort(mean_retention)   # most vulnerable first
sorted_emotions     = [valid_emotions[i] for i in sort_idx]
layer_sorted        = layer_ret_valid[:, sort_idx]
component_sorted    = component_ret_valid[:, sort_idx]

print(f'Valid emotions: {n_valid} (excluded {num_labels - n_valid} with baseline F1 <= 0.01)')
print(f'Excluded: {[emotion_names[i] for i in range(num_labels) if not valid_mask[i]]}')
print(f'\nMost vulnerable (lowest mean retention):')
for i in range(min(5, n_valid)):
    print(f'  {sorted_emotions[i]:18s}: {mean_retention[sort_idx[i]]:.3f}')
print(f'Most robust (highest mean retention):')
for i in range(-3, 0):
    print(f'  {sorted_emotions[i]:18s}: {mean_retention[sort_idx[i]]:.3f}')

## Part C — Visualizations

### Plot 1 — The Emotional Map (hero image)

In [ ]:
# Dual heatmap: component x emotion + layer x emotion
fig = plt.figure(figsize=(20, 16))
gs = fig.add_gridspec(2, 2, height_ratios=[6, 12], width_ratios=[50, 1],
                      hspace=0.28, wspace=0.03)
ax_comp  = fig.add_subplot(gs[0, 0])
ax_layer = fig.add_subplot(gs[1, 0])
cbar_ax  = fig.add_subplot(gs[:, 1])

vmin, vmax = 0.0, 1.0

im1 = ax_comp.imshow(component_sorted, cmap=RETENTION_CMAP, vmin=vmin, vmax=vmax,
                     aspect='auto', interpolation='nearest')
ax_comp.set_yticks(range(6))
ax_comp.set_yticklabels(COMPONENT_LABELS, fontsize=13, fontweight='bold')
for i, c in enumerate(COMP_COLORS):
    ax_comp.get_yticklabels()[i].set_color(c)
ax_comp.set_xticks(range(n_valid))
ax_comp.set_xticklabels(sorted_emotions, rotation=55, ha='right', fontsize=9)
ax_comp.set_title('COMPONENT LESION — compress one component across all layers',
                  fontsize=13, fontweight='bold', color=TEXT_SECONDARY, pad=10,
                  path_effects=[pe.withStroke(linewidth=2, foreground=DARK_BG)])

for i in range(6):
    for j in range(n_valid):
        val = component_sorted[i, j]
        txt_color = '#000000' if val > 0.55 else TEXT_PRIMARY
        ax_comp.text(j, i, f'{val:.0%}', ha='center', va='center',
                     fontsize=6.5, color=txt_color, fontweight='bold')

im2 = ax_layer.imshow(layer_sorted, cmap=RETENTION_CMAP, vmin=vmin, vmax=vmax,
                      aspect='auto', interpolation='nearest')
ax_layer.set_yticks(range(12))
ax_layer.set_yticklabels([f'Layer {i}' for i in range(12)], fontsize=12, fontweight='bold')
ax_layer.set_xticks(range(n_valid))
ax_layer.set_xticklabels(sorted_emotions, rotation=55, ha='right', fontsize=9)
ax_layer.set_title('LAYER LESION — compress all components of one layer',
                   fontsize=13, fontweight='bold', color=TEXT_SECONDARY, pad=10,
                   path_effects=[pe.withStroke(linewidth=2, foreground=DARK_BG)])

for i in range(12):
    for j in range(n_valid):
        val = layer_sorted[i, j]
        txt_color = '#000000' if val > 0.55 else TEXT_PRIMARY
        ax_layer.text(j, i, f'{val:.0%}', ha='center', va='center',
                      fontsize=5.5, color=txt_color)

cbar = fig.colorbar(im2, cax=cbar_ax)
cbar.set_label('F1 Retention', fontsize=14, color=TEXT_PRIMARY, labelpad=12)
cbar.ax.tick_params(colors=TEXT_SECONDARY, labelsize=11)
cbar.outline.set_edgecolor(DARK_GRID)

fig.suptitle('WHERE EMOTIONS LIVE INSIDE A TRANSFORMER',
             fontsize=26, fontweight='bold', color=TEXT_PRIMARY, y=0.99,
             path_effects=[pe.withStroke(linewidth=4, foreground=DARK_BG)])
fig.text(0.45, 0.965,
         f'Lesion study on BERT fine-tuned for GoEmotions  |  '
         f'Rank {LESION_RANK} compression  |  {n_valid} emotions',
         fontsize=11, color=TEXT_SECONDARY, ha='center',
         path_effects=[pe.withStroke(linewidth=2, foreground=DARK_BG)])

plt.savefig(os.path.join(out_dir, 'emotional_map_01_heatmap.png'),
            dpi=300, facecolor=DARK_BG, bbox_inches='tight', pad_inches=0.5)
plt.show()
print(f'Saved: {os.path.join(out_dir, "emotional_map_01_heatmap.png")}')

### Plot 2 — Emotional Genealogy (hierarchical clustering)

In [ ]:
# Build 18-dim sensitivity vector per emotion: [6 component + 12 layer] retentions
sensitivity_vectors = np.concatenate([
    component_ret_valid.T,   # (n_valid, 6)
    layer_ret_valid.T,       # (n_valid, 12)
], axis=1)  # (n_valid, 18)

dist_matrix = pdist(sensitivity_vectors, metric='cosine')
Z = linkage(dist_matrix, method='ward')
leaf_order = leaves_list(Z).astype(int)

# Also compute flat clusters for export (4 and 6 clusters)
cluster_labels_4 = fcluster(Z, t=4, criterion='maxclust')
cluster_labels_6 = fcluster(Z, t=6, criterion='maxclust')

fig, (ax_dend, ax_heat) = plt.subplots(
    2, 1, figsize=(18, 16),
    gridspec_kw={'height_ratios': [1, 2.5], 'hspace': 0.02},
)

set_link_color_palette(['#00ff87', '#00d4ff', '#7b2ff7', '#ff006e', '#ff9500', '#ffd60a'])

dend = dendrogram(
    Z,
    labels=valid_emotions,
    ax=ax_dend,
    color_threshold=0.6 * max(Z[:, 2]),
    leaf_rotation=90,
    leaf_font_size=11,
)
set_link_color_palette(None)

for spine in ax_dend.spines.values():
    spine.set_visible(False)
ax_dend.set_yticks([])
ax_dend.set_title(
    'EMOTIONAL GENEALOGY\nWhich emotions share the same "brain regions"?',
    fontsize=22, fontweight='bold', pad=15,
    path_effects=[pe.withStroke(linewidth=3, foreground=DARK_BG)])
ax_dend.tick_params(axis='x', colors=TEXT_PRIMARY, labelsize=10)

reordered_emotions = [valid_emotions[i] for i in leaf_order]
reordered_sens     = sensitivity_vectors[leaf_order]

dim_labels = COMPONENT_LABELS + [f'Layer {i}' for i in range(12)]

im = ax_heat.imshow(reordered_sens, cmap=RETENTION_CMAP, vmin=0, vmax=1,
                    aspect='auto', interpolation='nearest')
ax_heat.set_yticks(range(n_valid))
ax_heat.set_yticklabels(reordered_emotions, fontsize=10, fontweight='bold')
ax_heat.set_xticks(range(18))
ax_heat.set_xticklabels(dim_labels, fontsize=10, rotation=50, ha='right')

ax_heat.axvline(x=5.5, color=TEXT_PRIMARY, linewidth=2.5, alpha=0.6)
ax_heat.text(2.5, -1.5, 'Components', ha='center', fontsize=12,
             color=TEXT_SECONDARY, fontweight='bold',
             path_effects=[pe.withStroke(linewidth=2, foreground=DARK_BG)])
ax_heat.text(11.5, -1.5, 'Layers', ha='center', fontsize=12,
             color=TEXT_SECONDARY, fontweight='bold',
             path_effects=[pe.withStroke(linewidth=2, foreground=DARK_BG)])

for i in range(6):
    ax_heat.get_xticklabels()[i].set_color(COMP_COLORS[i])

cbar = fig.colorbar(im, ax=ax_heat, shrink=0.6, pad=0.02)
cbar.set_label('F1 Retention', fontsize=12, color=TEXT_PRIMARY)
cbar.ax.tick_params(colors=TEXT_SECONDARY)
cbar.outline.set_edgecolor(DARK_GRID)

plt.savefig(os.path.join(out_dir, 'emotional_map_02_genealogy.png'),
            dpi=300, facecolor=DARK_BG, bbox_inches='tight', pad_inches=0.5)
plt.show()
print(f'Saved: {os.path.join(out_dir, "emotional_map_02_genealogy.png")}')

### Plot 3 — Emotional Survival (extinction order)

In [ ]:
# Use existing per-emotion results from uniform compression sweep (notebooks 02/09)
csv_candidates = [
    os.path.join(results_dir, 'per_emotion_results.csv'),
    os.path.join(os.path.dirname(PROJECT_ROOT),
                 'TFG-Deliver', 'results', 'per_emotion_results.csv'),
]
df_existing = None
for p in csv_candidates:
    if os.path.exists(p):
        df_existing = pd.read_csv(p)
        print(f'Loaded: {p}')
        break

death_rank_records = []

if df_existing is None:
    print('WARNING: per_emotion_results.csv not found — skipping Plot 3')
else:
    strategies     = ['Baseline', 'Uniform r=512', 'Uniform r=384',
                      'Uniform r=256', 'Uniform r=128', 'Uniform r=64']
    strategy_short = ['Baseline', 'r=512', 'r=384', 'r=256', 'r=128', 'r=64']

    df_plot = df_existing[df_existing['strategy'].isin(strategies)].copy()

    bl_df = df_plot[df_plot['strategy'] == 'Baseline']
    valid_emos = bl_df[bl_df['f1'] > 0.01]['emotion'].tolist()
    df_plot = df_plot[df_plot['emotion'].isin(valid_emos)]

    pivot = df_plot.pivot(index='emotion', columns='strategy', values='f1')
    pivot = pivot[strategies]

    death_rank = {}
    for emo in valid_emos:
        bl = pivot.loc[emo, 'Baseline']
        death_rank[emo] = len(strategies)
        for si, strat in enumerate(strategies[1:], 1):
            if pivot.loc[emo, strat] < 0.05 * bl:
                death_rank[emo] = si
                break

    for emo in valid_emos:
        death_rank_records.append({
            'emotion': emo,
            'baseline_f1': pivot.loc[emo, 'Baseline'],
            'death_rank': death_rank[emo],
            'death_strategy': strategies[death_rank[emo]] if death_rank[emo] < len(strategies) else 'survives',
            'f1_final': pivot.loc[emo, strategies[-1]],
        })

    sorted_emos = sorted(valid_emos,
                         key=lambda e: (death_rank[e], pivot.loc[e, 'Baseline']))

    fig, ax = plt.subplots(figsize=(16, 10))
    x = np.arange(len(strategies))

    for emo in sorted_emos:
        values = [pivot.loc[emo, s] for s in strategies]
        dr = death_rank[emo]

        if   dr <= 1: color, alpha = '#ff006e', 0.85
        elif dr <= 2: color, alpha = '#ff4d6d', 0.75
        elif dr <= 3: color, alpha = '#ff9500', 0.70
        elif dr <= 4: color, alpha = '#ffd60a', 0.65
        else:         color, alpha = '#00ff87', 0.85

        lw = 2.5 if dr <= 2 or dr >= len(strategies) else 1.5
        ax.plot(x, values, '-o', color=color, alpha=alpha, linewidth=lw,
                markersize=4,
                path_effects=[pe.withStroke(linewidth=lw + 2,
                                           foreground=color + '20')])

    y_used = []
    for emo in sorted_emos:
        val = pivot.loc[emo, strategies[-1]]
        if val > 0.005:
            y_pos = val
            for yu in y_used:
                if abs(y_pos - yu) < 0.025:
                    y_pos = yu + 0.025
            y_used.append(y_pos)
            ax.annotate(emo, xy=(len(strategies) - 1, val),
                        xytext=(12, 0), textcoords='offset points',
                        fontsize=9, color='#00ff87', fontweight='bold',
                        va='center',
                        path_effects=[pe.withStroke(linewidth=2,
                                                   foreground=DARK_BG)])

    ax.set_xticks(x)
    ax.set_xticklabels(strategy_short, fontsize=14, fontweight='bold')
    ax.set_xlabel('Compression Level', fontsize=14, labelpad=10)
    ax.set_ylabel('F1 Score', fontsize=14, labelpad=10)
    ax.set_ylim(-0.03, 1.02)

    legend_items = [
        Line2D([0], [0], color='#ff006e', lw=2.5, label='Dies early (r=512-384)'),
        Line2D([0], [0], color='#ff9500', lw=2, label='Dies mid (r=256)'),
        Line2D([0], [0], color='#ffd60a', lw=2, label='Dies late (r=128)'),
        Line2D([0], [0], color='#00ff87', lw=2.5, label='Survives'),
    ]
    ax.legend(handles=legend_items, fontsize=12, loc='upper right',
              framealpha=0.15, edgecolor=DARK_GRID, labelcolor=TEXT_PRIMARY,
              fancybox=True)

    ax.set_title(
        'EMOTIONAL SURVIVAL\n'
        'The extinction order of emotions under compression',
        fontsize=22, fontweight='bold', pad=18,
        path_effects=[pe.withStroke(linewidth=3, foreground=DARK_BG)])

    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    ax.grid(True, axis='y', alpha=0.08, color='#ffffff')

    plt.savefig(os.path.join(out_dir, 'emotional_map_03_survival.png'),
                dpi=300, facecolor=DARK_BG, bbox_inches='tight', pad_inches=0.5)
    plt.show()
    print(f'Saved: {os.path.join(out_dir, "emotional_map_03_survival.png")}')

### Plot 4 — Emotional Brain Scan (4 representative emotions)

In [ ]:
# Select 4 emotions with distinct patterns
layer_var_per_emo  = layer_ret_valid.var(axis=0)
comp_var_per_emo   = component_ret_valid.var(axis=0)

candidates = []

idx_robust = int(np.argmax(mean_retention))
candidates.append((valid_emotions[idx_robust],
                   'ROBUST — survives everything',
                   '#00ff87'))

idx_fragile = int(np.argmin(mean_retention))
candidates.append((valid_emotions[idx_fragile],
                   'FRAGILE — first to collapse',
                   '#ff006e'))

layer_var_copy = layer_var_per_emo.copy()
layer_var_copy[idx_robust]  = -1
layer_var_copy[idx_fragile] = -1
idx_concentrated = int(np.argmax(layer_var_copy))
candidates.append((valid_emotions[idx_concentrated],
                   'CONCENTRATED — lives in specific layers',
                   '#7b2ff7'))

layer_var_copy2 = layer_var_per_emo.copy()
layer_var_copy2[idx_robust]       = np.inf
layer_var_copy2[idx_fragile]      = np.inf
layer_var_copy2[idx_concentrated] = np.inf
idx_distributed = int(np.argmin(layer_var_copy2))
candidates.append((valid_emotions[idx_distributed],
                   'DISTRIBUTED — spread across the network',
                   '#00d4ff'))

print('Selected emotions for brain scan:')
for emo, desc, _ in candidates:
    print(f'  {emo:18s}  {desc}')

fig, axes = plt.subplots(2, 2, figsize=(16, 16))

brainscan_records = []

for ax, (emotion, pattern_label, accent_color) in zip(axes.flat, candidates):
    emo_idx_full = emotion_names.index(emotion)

    layer_damage = 1.0 - layer_retention[:, emo_idx_full]
    comp_damage  = 1.0 - component_retention[:, emo_idx_full]

    grid = np.outer(layer_damage, comp_damage)
    if grid.max() > 0:
        grid = grid / grid.max()

    brainscan_records.append({
        'emotion': emotion,
        'pattern': pattern_label,
        'layer_variance': layer_var_per_emo[valid_emotions.index(emotion)],
        'component_variance': comp_var_per_emo[valid_emotions.index(emotion)],
        'mean_retention': mean_retention[valid_emotions.index(emotion)],
        'peak_layer': int(np.unravel_index(np.argmax(grid), grid.shape)[0]),
        'peak_component': COMPONENT_LABELS[int(np.unravel_index(np.argmax(grid), grid.shape)[1])],
    })

    im = ax.imshow(grid, cmap=NEON_CMAP, vmin=0, vmax=1,
                   aspect='auto', interpolation='nearest')

    ax.set_yticks(range(12))
    ax.set_yticklabels([f'L{i}' for i in range(12)], fontsize=9)
    ax.set_xticks(range(6))
    ax.set_xticklabels(['Q', 'K', 'V', 'AO', 'FFI', 'FFO'],
                       fontsize=11, fontweight='bold')
    for i, c in enumerate(COMP_COLORS):
        ax.get_xticklabels()[i].set_color(c)

    ax.set_title(f'{emotion.upper()}\n{pattern_label}',
                 fontsize=14, fontweight='bold', pad=12,
                 color=accent_color,
                 path_effects=[pe.withStroke(linewidth=2, foreground=DARK_BG)])

    ax.set_xticks(np.arange(-0.5, 6, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, 12, 1), minor=True)
    ax.grid(which='minor', color=DARK_BG, linewidth=1.5)
    ax.tick_params(which='minor', size=0)

    peak_pos = np.unravel_index(np.argmax(grid), grid.shape)
    ax.plot(peak_pos[1], peak_pos[0], 'o', markersize=18,
            markerfacecolor='none', markeredgecolor=TEXT_PRIMARY,
            markeredgewidth=2, alpha=0.8)

fig.suptitle('EMOTIONAL BRAIN SCAN\nWhere each emotion lives inside the transformer',
             fontsize=22, fontweight='bold', y=1.01,
             path_effects=[pe.withStroke(linewidth=3, foreground=DARK_BG)])
fig.text(0.5, 0.975,
         'Brighter = more critical  |  Circle = peak sensitivity  |  '
         'Rows = layers  |  Cols = components',
         fontsize=11, color=TEXT_SECONDARY, ha='center',
         path_effects=[pe.withStroke(linewidth=2, foreground=DARK_BG)])

cbar_ax2 = fig.add_axes([0.93, 0.15, 0.02, 0.7])
cbar = fig.colorbar(im, cax=cbar_ax2)
cbar.set_label('Importance (damage interaction)', fontsize=11, color=TEXT_PRIMARY)
cbar.ax.tick_params(colors=TEXT_SECONDARY)
cbar.outline.set_edgecolor(DARK_GRID)

plt.tight_layout(rect=[0, 0, 0.91, 0.96])
plt.savefig(os.path.join(out_dir, 'emotional_map_04_brainscan.png'),
            dpi=300, facecolor=DARK_BG, bbox_inches='tight', pad_inches=0.5)
plt.show()
print(f'Saved: {os.path.join(out_dir, "emotional_map_04_brainscan.png")}')

## Part D — Export

All CSVs are written to `results/notebook8/` for maximum information preservation.

In [ ]:
# 1. Layer lesion long-format results
rows_layer = []
for layer_idx in range(12):
    for i, emo in enumerate(emotion_names):
        rows_layer.append({
            'layer': layer_idx,
            'emotion': emo,
            'f1': layer_f1[layer_idx, i],
            'baseline_f1': baseline_per_emotion[i],
            'retention': layer_retention[layer_idx, i] if baseline_per_emotion[i] > 0 else np.nan,
        })
df_layer = pd.DataFrame(rows_layer)
p = os.path.join(out_dir, 'lesion_per_layer.csv')
df_layer.to_csv(p, index=False)
print(f'Saved: {p} ({len(df_layer)} rows)')

# 2. Component lesion long-format results
rows_comp = []
for ci, comp in enumerate(COMPONENT_LABELS):
    for i, emo in enumerate(emotion_names):
        rows_comp.append({
            'component': comp,
            'emotion': emo,
            'f1': component_f1[ci, i],
            'baseline_f1': baseline_per_emotion[i],
            'retention': component_retention[ci, i] if baseline_per_emotion[i] > 0 else np.nan,
        })
df_comp = pd.DataFrame(rows_comp)
p = os.path.join(out_dir, 'lesion_per_component.csv')
df_comp.to_csv(p, index=False)
print(f'Saved: {p} ({len(df_comp)} rows)')

In [ ]:
# 3. Summary: baseline + all lesion F1 macros
rows_summary = []
rows_summary.append({'experiment': 'baseline', 'target': 'none',
                     'f1_macro': baseline_f1_macro})
for layer_idx in range(12):
    rows_summary.append({'experiment': 'layer_lesion', 'target': f'layer_{layer_idx}',
                         'f1_macro': layer_f1_macro[layer_idx]})
for ci, comp in enumerate(COMPONENT_LABELS):
    rows_summary.append({'experiment': 'component_lesion', 'target': comp,
                         'f1_macro': component_f1_macro[ci]})
df_summary = pd.DataFrame(rows_summary)
p = os.path.join(out_dir, 'lesion_summary.csv')
df_summary.to_csv(p, index=False)
print(f'Saved: {p} ({len(df_summary)} rows)')

# 4. Layer retention matrix (wide format)
df_layer_matrix = pd.DataFrame(
    layer_retention,
    index=[f'layer_{i}' for i in range(12)],
    columns=emotion_names,
)
df_layer_matrix.index.name = 'target'
p = os.path.join(out_dir, 'layer_retention_matrix.csv')
df_layer_matrix.to_csv(p)
print(f'Saved: {p}')

# 5. Component retention matrix (wide format)
df_comp_matrix = pd.DataFrame(
    component_retention,
    index=COMPONENT_LABELS,
    columns=emotion_names,
)
df_comp_matrix.index.name = 'target'
p = os.path.join(out_dir, 'component_retention_matrix.csv')
df_comp_matrix.to_csv(p)
print(f'Saved: {p}')

In [ ]:
# 6. Sensitivity vectors (18-dim per emotion) used for clustering
df_sens = pd.DataFrame(
    sensitivity_vectors,
    index=valid_emotions,
    columns=COMPONENT_LABELS + [f'Layer_{i}' for i in range(12)],
)
df_sens.index.name = 'emotion'
p = os.path.join(out_dir, 'sensitivity_vectors.csv')
df_sens.to_csv(p)
print(f'Saved: {p} ({df_sens.shape[0]} emotions x {df_sens.shape[1]} dims)')

# 7. Emotional genealogy clusters (4 and 6 flat clusters + dendrogram leaf order)
leaf_rank = {valid_emotions[leaf_order[k]]: k for k in range(len(leaf_order))}
df_clusters = pd.DataFrame({
    'emotion': valid_emotions,
    'mean_retention': mean_retention,
    'cluster_k4': cluster_labels_4,
    'cluster_k6': cluster_labels_6,
    'leaf_order': [leaf_rank[e] for e in valid_emotions],
})
p = os.path.join(out_dir, 'emotional_genealogy_clusters.csv')
df_clusters.to_csv(p, index=False)
print(f'Saved: {p}')

# 8. Linkage matrix (raw Ward linkage Z)
df_linkage = pd.DataFrame(Z, columns=['cluster_a', 'cluster_b', 'distance', 'n_members'])
p = os.path.join(out_dir, 'linkage_matrix.csv')
df_linkage.to_csv(p, index=False)
print(f'Saved: {p}')

In [ ]:
# 9. Death rank (extinction order) — skipped if external CSV missing
if death_rank_records:
    df_death = pd.DataFrame(death_rank_records)
    p = os.path.join(out_dir, 'emotion_death_rank.csv')
    df_death.to_csv(p, index=False)
    print(f'Saved: {p} ({len(df_death)} rows)')
else:
    print('Skipped emotion_death_rank.csv (no external survival data)')

# 10. Brain scan selection metadata
df_brainscan = pd.DataFrame(brainscan_records)
p = os.path.join(out_dir, 'brainscan_selected_emotions.csv')
df_brainscan.to_csv(p, index=False)
print(f'Saved: {p}')

# 11. Final run summary
print('\n' + '=' * 65)
print('LESION STUDY COMPLETE')
print('=' * 65)
print(f'Baseline F1 macro: {baseline_f1_macro:.4f}')
print(f'Lesion rank: {LESION_RANK}')
print(f'\nPer-layer lesion (F1 macro):')
for i in range(12):
    bar = '#' * int(layer_f1_macro[i] / baseline_f1_macro * 40)
    print(f'  Layer {i:2d}: {layer_f1_macro[i]:.4f}  '
          f'({layer_f1_macro[i]/baseline_f1_macro*100:5.1f}%)  {bar}')
print(f'\nPer-component lesion (F1 macro):')
for i, comp in enumerate(COMPONENT_LABELS):
    bar = '#' * int(component_f1_macro[i] / baseline_f1_macro * 40)
    print(f'  {comp:12s}: {component_f1_macro[i]:.4f}  '
          f'({component_f1_macro[i]/baseline_f1_macro*100:5.1f}%)  {bar}')
print(f'\nFigures saved in: {out_dir}/')
for fig_name in ['emotional_map_01_heatmap.png', 'emotional_map_02_genealogy.png',
                 'emotional_map_03_survival.png', 'emotional_map_04_brainscan.png']:
    path = os.path.join(out_dir, fig_name)
    status = 'ok' if os.path.exists(path) else 'MISSING'
    print(f'  [{status}] {fig_name}')